# CS781 Phase 3 - Rhetorical Roles - notebook `extra`

**Full design:** sentences (Phase-1 tokenization, <=128 tokens) packed into 512-token windows -> Phase-1 backbone ->
mean-pooled sentence vectors + relative position (16 sinusoids) + CL/IT embedding -> BiLSTM over a range of ~70-100
consecutive sentences (3,072 tokens) -> 13 labels; cross-entropy with sqrt class weights (cap 10).
**Base technique:** LoRA r16 (lr 5e-4). Experiments 15-25 (design ablations, seeds) each change one thing from the base.
Training on CL_train + IT_train only; the epoch (max 10, patience 4) is chosen on CL_dev + IT_dev with the official metric
(macro-F1 over all 13 labels). Dev and test are never trained on.

| ID | GPU | run | technique | change vs full design | question |
|---|---|---|---|---|---|
| B22 | GPU0 | `B22_lora_abl_bilstm2` | LoRA r16 | 2-layer BiLSTM | head capacity |
| B24 | GPU0 | `B24_lora_seed7` | LoRA r16 | seed 7 | run-to-run variance |
| C24 | GPU0 | `C24_lora_seed7` | LoRA r16 | seed 7 | run-to-run variance |
| D17 | GPU0 | `D17_lora_abl_singlebilstm` | LoRA r16 | sentence alone + BiLSTM (range 1536) | sentence-level context only |
| D20 | GPU0 | `D20_lora_abl_lastpool` | LoRA r16 | last-token pooling | sentence representation |
| D22 | GPU0 | `D22_lora_abl_bilstm2` | LoRA r16 | 2-layer BiLSTM | head capacity |
| D24 | GPU0 | `D24_lora_seed7` | LoRA r16 | seed 7 | run-to-run variance |
| B23 | GPU1 | `B23_lora_abl_nodom` | LoRA r16 | no domain embedding | value of the CL/IT signal |
| B25 | GPU1 | `B25_lora_seed11` | LoRA r16 | seed 11 | run-to-run variance |
| C25 | GPU1 | `C25_lora_seed11` | LoRA r16 | seed 11 | run-to-run variance |
| D16 | GPU1 | `D16_lora_abl_windowonly` | LoRA r16 | packed windows, MLP head | token-level context only |
| D18 | GPU1 | `D18_lora_abl_nopos` | LoRA r16 | no position features | value of the position prior |
| D19 | GPU1 | `D19_lora_abl_noweights` | LoRA r16 | no class weights | class imbalance handling |
| D21 | GPU1 | `D21_lora_abl_ctx6144` | LoRA r16 | range 6144 tokens | longer document context |
| D23 | GPU1 | `D23_lora_abl_nodom` | LoRA r16 | no domain embedding | value of the CL/IT signal |
| D25 | GPU1 | `D25_lora_seed11` | LoRA r16 | seed 11 | run-to-run variance |

**Inputs:** dataset `cs781-p3-code` (latest version), `cs781-legal-data`, Phase-1 checkpoint of model B + C + D.
Internet ON, secret `HF_TOKEN`, GPU **T4 x2**. Save Version -> Save & Run All.
Expected: ~2.5 h after a 3-5 min smoke test. A run that fails the smoke test is dropped (its error is in the
report); the others continue.
**Outputs:** `submissions/<run>.zip` (ready to submit), `models/<run>.pt` (trained parameters + config + best epoch +
dev score), `bundle_extra.zip` (dev/test probabilities + reports, for the ensemble), `REPORT_extra.txt`.

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS, MODELS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs", f"{W}/models"
CACHE = f"{W}/rr_cache.pkl"
for d in (RUNS, MODELS, f"{W}/logs", f"{W}/submissions"):
    os.makedirs(d, exist_ok=True)

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", sorted(f for f in os.listdir(WORK) if f.endswith(".py")))

# ---- Phase-3 data from Hugging Face
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
print("p3 data:", sorted(os.listdir(P3DATA)))

# dev truth files in the official evaluator's format (used to cross-check our metric)
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))

# ---- Phase-1 checkpoints (final.pt per model letter; continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

# ---- tokenize every split once (shared by all jobs) + CPU tests
t0 = time.time()
cache = rr_data.build_cache(P3DATA, TOK_ROOT, CACHE)
for k, v in cache["docs"].items():
    print(k, len(v), "docs", sum(len(x["sents"]) for x in v), "sentences")
print(f"cache built in {time.time() - t0:.0f}s")
r = subprocess.run("python tests_rr.py", shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed"

code: ['adapt.py', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'evaluate_phase3.py', 'gated_deltanet.py', 'kv_cache.py', 'layers.py', 'models.py', 'ner_data.py', 'ner_model.py', 'p1_loader.py', 'peft.py', 'rr_data.py', 'rr_ensemble.py', 'rr_eval.py', 'rr_model.py', 'rr_train.py', 'tests_rr.py']


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

p3 data: ['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']
  model b: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-b/pytorch/continual-train-9385-3615/1/run_b_gpt_mha_cpt/final.pt
  model c: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-c/pytorch/phase-1/1/run_c_local_gqa_global_mha/final.pt
  model d: /kaggle/input/models/mrinaldevnath/cs781-capstone-model-d/pytorch/continual-train-v2/1/run_d_gdn_mla_cpt/final.pt
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

('CL', 'train') 40 docs 10593 sentences
('CL', 'dev') 5 docs 1396 sentences
('CL', 'test') 5 docs 1339 sentences
('IT', 'train') 40 docs 6252 sentences
('IT', 'dev') 5 docs 746 sentences
('IT', 'test') 5 docs 858 sentences
cache built in 3s
ok test_segments
ok test_pack_and_arrays
ok test_metric
ok test_files_and_validation
 


In [2]:
def cmd(model, out, extra, model_dir):
    return (f"python rr_train.py --ckpt {CKPT[model]} --tok_root {TOK_ROOT} --data_dir {P3DATA} --cache {CACHE} "
            f"--out {out} --model_dir {model_dir} {extra}")

def run(tag, c, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    with open(f"{W}/logs/{tag}.txt", "w") as f:
        p = subprocess.run(c, shell=True, cwd=WORK, env=env, stdout=f, stderr=subprocess.STDOUT)
    tail = open(f"{W}/logs/{tag}.txt").read().strip().splitlines()[-2:]
    print(f"[GPU{gpu}] {tag}: exit {p.returncode} | " + " / ".join(tail)[-300:], flush=True)
    return p.returncode

def run_queues(queues, smoke=False):
    """queues: one list of experiments per GPU, run in parallel. smoke: 1 tiny epoch per experiment."""
    status = {}
    def worker(gpu, jobs):
        for e in jobs:
            t = ("smoke_" if smoke else "") + e["tag"]
            extra = e["args"] + (" --max_docs 1 --max_sents 40 --epochs 1" if smoke else "")
            if smoke and e is not jobs[0]:
                extra += " --no_test"
            status[t] = run(t, cmd(e["model"], f"{RUNS}/{t}", extra, f"{W}/smoke_models" if smoke else MODELS), gpu)
    th = [threading.Thread(target=worker, args=(g, q)) for g, q in enumerate(queues) if q]
    t0 = time.time()
    for t in th: t.start()
    for t in th: t.join()
    print(f"{'smoke' if smoke else 'runs'} done in {(time.time() - t0) / 60:.1f} min:", status)
    return status

def smoke(queues):
    """Runs 1 tiny epoch of every experiment. A failing experiment is dropped (its error is printed and
    reported) instead of stopping the notebook; the notebook stops only if every experiment fails."""
    st = run_queues(queues, smoke=True)
    bad = [k for k, v in st.items() if v != 0]
    for k in bad:
        print(f"===== SMOKE FAILED, dropped: {k}\n" + open(f"{W}/logs/{k}.txt").read()[-3000:])
    assert len(bad) < len(st), "every experiment failed the smoke test"
    for k in st:
        shutil.rmtree(f"{RUNS}/{k}", ignore_errors=True)
    shutil.rmtree(f"{W}/smoke_models", ignore_errors=True)
    print("SMOKE OK" if not bad else f"SMOKE OK except {len(bad)} dropped: {bad}")
    return [[e for e in q if "smoke_" + e["tag"] not in bad] for q in queues]

## Experiments (one queue per GPU)

In [3]:
NAME = 'extra'
QUEUES = [[{'id': 'B22',
   'tag': 'B22_lora_abl_bilstm2',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': '2-layer BiLSTM',
   'args': '--method lora --rank 16 --alpha 32 --rnn_layers 2'},
  {'id': 'B24', 'tag': 'B24_lora_seed7', 'model': 'b', 'tech': 'LoRA r16', 'change': 'seed 7', 'args': '--method lora --rank 16 --alpha 32 --seed 7'},
  {'id': 'C24', 'tag': 'C24_lora_seed7', 'model': 'c', 'tech': 'LoRA r16', 'change': 'seed 7', 'args': '--method lora --rank 16 --alpha 32 --seed 7'},
  {'id': 'D17',
   'tag': 'D17_lora_abl_singlebilstm',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'sentence alone + BiLSTM (range 1536)',
   'args': '--method lora --rank 16 --alpha 32 --mode single --seg_tokens 1536'},
  {'id': 'D20',
   'tag': 'D20_lora_abl_lastpool',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'last-token pooling',
   'args': '--method lora --rank 16 --alpha 32 --pool last'},
  {'id': 'D22',
   'tag': 'D22_lora_abl_bilstm2',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': '2-layer BiLSTM',
   'args': '--method lora --rank 16 --alpha 32 --rnn_layers 2'},
  {'id': 'D24',
   'tag': 'D24_lora_seed7',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'seed 7',
   'args': '--method lora --rank 16 --alpha 32 --seed 7'}],
 [{'id': 'B23',
   'tag': 'B23_lora_abl_nodom',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'no domain embedding',
   'args': '--method lora --rank 16 --alpha 32 --no_dom'},
  {'id': 'B25',
   'tag': 'B25_lora_seed11',
   'model': 'b',
   'tech': 'LoRA r16',
   'change': 'seed 11',
   'args': '--method lora --rank 16 --alpha 32 --seed 11'},
  {'id': 'C25',
   'tag': 'C25_lora_seed11',
   'model': 'c',
   'tech': 'LoRA r16',
   'change': 'seed 11',
   'args': '--method lora --rank 16 --alpha 32 --seed 11'},
  {'id': 'D16',
   'tag': 'D16_lora_abl_windowonly',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'packed windows, MLP head',
   'args': '--method lora --rank 16 --alpha 32 --head mlp'},
  {'id': 'D18',
   'tag': 'D18_lora_abl_nopos',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'no position features',
   'args': '--method lora --rank 16 --alpha 32 --no_pos'},
  {'id': 'D19',
   'tag': 'D19_lora_abl_noweights',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'no class weights',
   'args': '--method lora --rank 16 --alpha 32 --class_weight none'},
  {'id': 'D21',
   'tag': 'D21_lora_abl_ctx6144',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'range 6144 tokens',
   'args': '--method lora --rank 16 --alpha 32 --seg_tokens 6144'},
  {'id': 'D23',
   'tag': 'D23_lora_abl_nodom',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'no domain embedding',
   'args': '--method lora --rank 16 --alpha 32 --no_dom'},
  {'id': 'D25',
   'tag': 'D25_lora_seed11',
   'model': 'd',
   'tech': 'LoRA r16',
   'change': 'seed 11',
   'args': '--method lora --rank 16 --alpha 32 --seed 11'}]]
ALL_QUEUES = [list(q) for q in QUEUES]                    # the full plan (the report lists dropped runs too)

## 1. Smoke test

In [4]:
QUEUES = smoke(QUEUES)

[GPU1] smoke_B23_lora_abl_nodom: exit 0 | sion.zip  {'Fact': 2197, 'Issue': 0, 'ArgumentPetitioner': 0, 'ArgumentRespondent': 0, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 0, 'RatioOfTheDecision': 0, 'RulingByPresentCourt': 0, 'None': 0} / [06:35:37] DONE
[GPU0] smoke_B22_lora_abl_bilstm2: exit 0 | sion.zip  {'Fact': 2197, 'Issue': 0, 'ArgumentPetitioner': 0, 'ArgumentRespondent': 0, 'Statute': 0, 'Dissent': 0, 'PrecedentReliedUpon': 0, 'PrecedentNotReliedUpon': 0, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 0, 'RatioOfTheDecision': 0, 'RulingByPresentCourt': 0, 'None': 0} / [06:35:37] DONE
[GPU0] smoke_B24_lora_seed7: exit 0 | confusions ArgumentPetitioner->Fact:16 RatioOfTheDecision->Fact:16 RulingByLowerCourt->Fact:4 None->Fact:3 Issue->Fact:3 Statute->Fact:1 / [06:35:47] DONE
[GPU1] smoke_B25_lora_seed11: exit 0 | confusions ArgumentPetitioner->Fact:16 RatioOfTheDecision->Fact:16 RulingByLower

## 2. Runs

In [5]:
status = run_queues(QUEUES)

[GPU0] B22_lora_abl_bilstm2: exit 0 | ct': 431, 'Issue': 16, 'ArgumentPetitioner': 215, 'ArgumentRespondent': 85, 'Statute': 177, 'Dissent': 3, 'PrecedentReliedUpon': 333, 'PrecedentNotReliedUpon': 52, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 136, 'RatioOfTheDecision': 683, 'RulingByPresentCourt': 38, 'None': 28} / [06:49:24] DONE
[GPU1] B23_lora_abl_nodom: exit 0 | ct': 397, 'Issue': 10, 'ArgumentPetitioner': 264, 'ArgumentRespondent': 54, 'Statute': 168, 'Dissent': 4, 'PrecedentReliedUpon': 380, 'PrecedentNotReliedUpon': 24, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 104, 'RatioOfTheDecision': 727, 'RulingByPresentCourt': 45, 'None': 20} / [06:50:44] DONE
[GPU0] B24_lora_seed7: exit 0 | t': 419, 'Issue': 15, 'ArgumentPetitioner': 255, 'ArgumentRespondent': 49, 'Statute': 170, 'Dissent': 30, 'PrecedentReliedUpon': 328, 'PrecedentNotReliedUpon': 26, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 129, 'RatioOfTheDecision': 711, 'RulingByPresentCourt': 46, 'None': 19} / [07:01:

## 3. Report

In [6]:
SHORT = {"ArgumentPetitioner": "ArgPet", "ArgumentRespondent": "ArgResp", "PrecedentReliedUpon": "PrecRel",
         "PrecedentNotReliedUpon": "PrecNRel", "PrecedentOverruled": "PrecOvr", "RulingByLowerCourt": "RulLower",
         "RatioOfTheDecision": "Ratio", "RulingByPresentCourt": "RulPres"}

def official(tag):
    """Official evaluator on this run's dev predictions (cross-check of our metric)."""
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {RUNS}/{tag}/dev_pred.json --labels {P3DATA}/label_vocab.json",
                       shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")

def log_tail(name, n=2):
    p = f"{W}/logs/{name}.txt"
    return " / ".join(open(p).read().strip().splitlines()[-n:])[-250:] if os.path.exists(p) else ""

def summarize(name, queues):
    exps = [e for q in queues for e in q]
    L = [f"=== REPORT {name} ===",
         "ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem"]
    rows = []
    for e in exps:
        p = f"{RUNS}/{e['tag']}/summary.json"
        if not os.path.exists(p):
            why = "SMOKE FAILED (dropped)" if not os.path.exists(f"{W}/logs/{e['tag']}.txt") else "FAILED"
            L.append(f"{e['id']} | {e['tag']:28s} | {why}: {log_tail(e['tag']) or log_tail('smoke_' + e['tag'])}")
            continue
        s = json.load(open(p)); dv = s["dev"]
        off = official(e["tag"])
        rows.append((e, s))
        L.append(f"{e['id']} | {e['tag']:28s} | {e['tech'][:18]:18s} | {e['change'][:26]:26s} | {dv['macro_f1']:.4f} | {off:.4f}   | "
                 f"{dv['macro_f1_CL']:.4f} | {dv['macro_f1_IT']:.4f} | {dv['macro_p']:.4f} | {dv['macro_r']:.4f} | {s['best_epoch']:2d} | "
                 f"{s['params_trainable'] / 1e6:8.2f}M | {s['train_minutes']:4.0f} | {s.get('tok_s') or 0:6,.0f} | {s.get('peak_mem_gb') or 0:.1f}G")
        shutil.copy(f"{RUNS}/{e['tag']}/submission.zip", f"{W}/submissions/{e['tag']}.zip")
    if rows:
        labs = rows[0][1]["labels"]
        L.append("\nper-label dev F1 (official: all 13 labels count; Dissent and PrecedentOverruled have no dev sentences -> 0)")
        L.append(f"{'ID':4s} " + " ".join(f"{SHORT.get(l, l)[:8]:>8s}" for l in labs))
        for e, s in rows:
            pl = s["dev"]["per_label"]
            L.append(f"{e['id']:4s} " + " ".join(f"{pl[l]['f1']:8.2f}" for l in labs))
        L.append("\ntop confusions (gold->pred:count)")
        for e, s in rows:
            L.append(f"{e['id']}: " + " ".join(s["dev"]["top_confusions"][:6]))
    bundle = f"{W}/bundle_{name}.zip"                     # everything except model weights (those stay in models/)
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for e in exps:
            d = f"{RUNS}/{e['tag']}"
            if os.path.isdir(d):
                for f in os.listdir(d):
                    z.write(f"{d}/{f}", arcname=f"{e['tag']}/{f}")
    L.append(f"\nbundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB)")
    L.append("models: " + ", ".join(f"{f} ({os.path.getsize(f'{MODELS}/{f}') / 2**20:.0f} MB)" for f in sorted(os.listdir(MODELS))))
    L.append("submission files: " + ", ".join(sorted(os.listdir(f"{W}/submissions"))))
    rep = "\n".join(L)
    open(f"{W}/REPORT_{name}.txt", "w").write(rep)
    print(rep + "\n=== END REPORT ===")

summarize(NAME, ALL_QUEUES)

=== REPORT extra ===
ID  | run                          | technique          | change                     | dev F1 | official | CL     | IT     | P      | R      | ep | trainable | min  | tok/s  | mem
B22 | B22_lora_abl_bilstm2         | LoRA r16           | 2-layer BiLSTM             | 0.4945 | 0.4945   | 0.3910 | 0.5160 | 0.5250 | 0.5030 |  5 |     6.26M |   12 |  8,603 | 3.5G
B24 | B24_lora_seed7               | LoRA r16           | seed 7                     | 0.5285 | 0.5285   | 0.4430 | 0.5151 | 0.5495 | 0.5238 |  5 |     4.68M |   12 |  8,757 | 3.4G
C24 | C24_lora_seed7               | LoRA r16           | seed 7                     | 0.5037 | 0.5037   | 0.3914 | 0.5362 | 0.5533 | 0.4874 |  8 |     4.64M |   13 |  8,794 | 3.4G
D17 | D17_lora_abl_singlebilstm    | LoRA r16           | sentence alone + BiLSTM (r | 0.4997 | 0.4997   | 0.3865 | 0.5161 | 0.5458 | 0.4859 |  4 |     5.00M |   37 |  2,547 | 10.1G
D20 | D20_lora_abl_lastpool        | LoRA r16           | last-token pooli